# Data Exploration

Profiling `movies_main.csv`, `ratings.json`, and `movie_extended.csv` ahead of Task 5 (cleaning) and Task 6 (validation). Not graded directly — this is process evidence; the written findings at the bottom are the real deliverable.

In [1]:
import json
import re
from pathlib import Path

import pandas as pd

RAW_DIR = Path("../data/raw")

movies_main = pd.read_csv(RAW_DIR / "movies_main.csv")

with open(RAW_DIR / "ratings.json") as f:
    ratings_raw = json.load(f)

movie_extended = pd.read_csv(RAW_DIR / "movie_extended.csv")

print("movies_main:", movies_main.shape)
print("ratings.json records:", len(ratings_raw))
print("movie_extended:", movie_extended.shape)

movies_main: (45486, 5)
ratings.json records: 9067
movie_extended: (45466, 5)


## 1. `movies_main.csv`

Checking the known issues: duplicates, mixed date formats, missing budget.

In [2]:
# Duplicates: exact rows, by id, by title alone, and by title+release_date
# (title alone is too loose -- remakes/shorts legitimately share titles)
print("exact duplicate rows:", movies_main.duplicated().sum())
print("duplicate id:", movies_main.duplicated(subset=["id"]).sum())
print("duplicate title only:", movies_main.duplicated(subset=["title"]).sum())
print(
    "duplicate title + release_date:",
    movies_main.duplicated(subset=["title", "release_date"]).sum(),
)

movies_main[movies_main.duplicated(subset=["title", "release_date"], keep=False)].sort_values(
    "title"
).head(10)

exact duplicate rows: 6
duplicate id: 50
duplicate title only: 3188
duplicate title + release_date: 13


,id,title,release_date,budget,revenue
11155,77221,Black Gold,21-12-2011,NaN,5446000.0
20843,77221,Black Gold,21-12-2011,40000000,5446000.0
17229,25541,Brotherhood,21-10-2009,0,0.0
23044,25541,Brotherhood,21-10-2009,0,0.0
5865,4912,Confessions of a Dangerous Mind,12/30/2002,30000000,33013805.0
33826,4912,Confessions of a Dangerous Mind,12/30/2002,NaN,33013805.0
14000,18440,Days of Darkness,2007-01-01,0,0.0
22151,18440,Days of Darkness,2007-01-01,0,0.0
7345,5511,Le Samouraï,1967-10-25,0,39481.0
9165,5511,Le Samouraï,1967-10-25,0,39481.0


In [3]:
# Also flag the 3 rows where `id` itself isn't numeric -- a sign the whole
# row's columns got shifted (see what lands in `title`/`budget` below).
non_numeric_id = movies_main[~movies_main["id"].astype(str).str.match(r"^\d+$")]
print("rows with non-numeric id:", len(non_numeric_id))
non_numeric_id

rows with non-numeric id: 3


,id,title,release_date,budget,revenue
19730,1997-08-20,NaN,NaN,/ff9qCepilowshEtG2GYWwzt2bs4.jpg,NaN
29503,2012-09-29,NaN,NaN,0,NaN
35587,2014-01-01,NaN,NaN,/zaSf5OG7V8X8gqFvly88zDdRm46.jpg,NaN


In [4]:
# Date formats: sample raw strings and classify by pattern rather than
# letting pd.to_datetime silently guess (it mis-parses ambiguous DD/MM vs MM/DD).
def classify_date(s: str) -> str:
    if re.match(r"^\d{4}-\d{2}-\d{2}$", s):
        return "YYYY-MM-DD"
    if re.match(r"^\d{1,2}/\d{1,2}/\d{4}$", s):
        return "MM/DD/YYYY"
    if re.match(r"^\d{1,2}-\d{1,2}-\d{4}$", s):
        return "DD-MM-YYYY"
    return f"other: {s}"


print("null release_date:", movies_main["release_date"].isna().sum())
date_formats = movies_main["release_date"].dropna().astype(str).apply(classify_date)
date_formats.value_counts()

null release_date: 90


release_date
YYYY-MM-DD    15194
DD-MM-YYYY    15135
MM/DD/YYYY    15067
Name: count, dtype: int64

In [5]:
# Budget: `budget` reads in as string because of the shifted rows above,
# so coerce to numeric and check both true nulls and 0-as-placeholder.
budget_num = pd.to_numeric(movies_main["budget"], errors="coerce")
n = len(movies_main)
print(f"true null budget: {budget_num.isna().sum()} ({budget_num.isna().mean():.1%})")
print(f"budget == 0:       {(budget_num == 0).sum()} ({(budget_num == 0).mean():.1%})")
missing_or_zero = budget_num.isna() | (budget_num == 0)
print(f"null + zero combined: {missing_or_zero.sum()} ({missing_or_zero.mean():.1%})")

true null budget: 4437 (9.8%)


budget == 0:       33887 (74.5%)
null + zero combined: 38324 (84.3%)


## 2. `ratings.json`

Look at the raw shape before touching pandas.

In [6]:
print(type(ratings_raw), len(ratings_raw))
ratings_raw[0]

<class 'list'> 9067


{'movie_id': 326305,
 'ratings_summary': {'avg_rating': 3.8724696356275303,
  'total_ratings': 247,
  'std_dev': 0.9589814920649079},
 'last_rated': 1475783711}

In [7]:
# It's a flat list of one dict per movie (not per rating event), with a single
# nested `ratings_summary` object. Confirm the key shape and movie_id uniqueness.
key_shapes = {tuple(sorted(r.keys())) for r in ratings_raw}
movie_ids = [r["movie_id"] for r in ratings_raw]
print("distinct top-level key shapes:", key_shapes)
print("records:", len(movie_ids), "unique movie_id:", len(set(movie_ids)))

# Exploration only -- real flattening logic belongs in Task 5's data_processor.py.
pd.json_normalize(ratings_raw, sep="_").head(3)

distinct top-level key shapes: {('last_rated', 'movie_id', 'ratings_summary')}
records: 9067 unique movie_id: 9067


,movie_id,last_rated,ratings_summary_avg_rating,ratings_summary_total_ratings,ratings_summary_std_dev
0,326305,1475783711,3.872470,247,0.958981
1,15850,1470073353,3.401869,107,0.880714
2,2925,1471385241,3.161017,59,1.150115


## 3. `movie_extended.csv`

How each multi-valued field is encoded, since that determines the junction-table split for Task 3's schema.

In [8]:
# genres / production_companies: plain comma-separated strings
print("genres sample:", movie_extended["genres"].dropna().iloc[0])
print("production_companies sample:", movie_extended["production_companies"].dropna().iloc[0])
print("genres contain ';':", movie_extended["genres"].dropna().str.contains(";").sum())
print()
print("null counts:")
movie_extended.isna().sum()

genres sample: Animation,Comedy,Family
production_companies sample: Pixar Animation Studios


genres contain ';': 0

null counts:


id                          0
genres                   2442
production_companies    12365
production_countries        0
spoken_languages            6
dtype: int64

In [9]:
import ast

# production_countries / spoken_languages: Python-literal (single-quoted) list-of-dicts,
# NOT valid JSON -- needs ast.literal_eval, not json.loads.
sample = movie_extended["production_countries"].dropna().iloc[1]
print("raw string:", sample)
print("parsed:", ast.literal_eval(sample))

empty_countries = movie_extended["production_countries"].astype(str).str.strip() == "[]"
print("\nrows with empty production_countries list:", empty_countries.sum())

raw string: [{'iso_3166_1': 'US', 'name': 'United States of America'}]
parsed: [{'iso_3166_1': 'US', 'name': 'United States of America'}]

rows with empty production_countries list: 6290


## 4. Cross-file sanity checks

Join key consistency and orphaned records across all three files.

In [10]:
# `id` is the shared join key across all three files (movies_main.id,
# movie_extended.id, ratings.json[].movie_id). Exclude the 3 corrupt-id rows
# found above -- they can't meaningfully join to anything by definition.
main_ids = set(movies_main["id"].astype(str)) - set(non_numeric_id["id"].astype(str))
ext_ids = set(movie_extended["id"].astype(str)) - set(non_numeric_id["id"].astype(str))
ratings_ids = set(str(r["movie_id"]) for r in ratings_raw)

print("movies_main <-> movie_extended: perfectly aligned id sets:", main_ids == ext_ids)
print("ratings ids not present in movies_main (orphaned ratings):", len(ratings_ids - main_ids))
print(
    "movies_main ids with NO ratings entry:",
    len(main_ids - ratings_ids),
    f"({len(main_ids - ratings_ids) / len(main_ids):.1%} of catalog)",
)

movies_main <-> movie_extended: perfectly aligned id sets: True
ratings ids not present in movies_main (orphaned ratings): 0
movies_main ids with NO ratings entry: 36366 (80.0% of catalog)


In [11]:
# Shortlist: movies with a suspiciously high rating paired with a missing/zero
# budget -- candidates for Task 6 external validation (hidden gem vs. bad data).
ratings_df = pd.DataFrame(
    {
        "movie_id": [r["movie_id"] for r in ratings_raw],
        "avg_rating": [r["ratings_summary"]["avg_rating"] for r in ratings_raw],
        "total_ratings": [r["ratings_summary"]["total_ratings"] for r in ratings_raw],
    }
)

candidates = movies_main.copy()
candidates["id"] = pd.to_numeric(candidates["id"], errors="coerce")
candidates["budget_num"] = budget_num
merged = candidates.merge(ratings_df, left_on="id", right_on="movie_id")

suspicious = merged[
    (merged["budget_num"].fillna(0) <= 100_000)
    & (merged["avg_rating"] >= 4.2)
    & (merged["total_ratings"] >= 50)
].sort_values("avg_rating", ascending=False)

print(f"{len(suspicious)} candidates: high rating (>=4.2, >=50 ratings), budget <= $100k")
suspicious[["id", "title", "budget_num", "avg_rating", "total_ratings"]].head(8)

30 candidates: high rating (>=4.2, >=50 ratings), budget <= $100k


,id,title,budget_num,avg_rating,total_ratings
245,1554.0,Down by Law,0.0,4.487138,311
125,123505.0,A Modern Affair,0.0,4.420000,50
5200,41275.0,Python,0.0,4.387097,62
2603,112912.0,Why We Fight: The Battle of Russia,0.0,4.385185,135
2327,175027.0,Double Harness,0.0,4.370647,201
5498,93904.0,Snowballs,0.0,4.350000,50
8651,345906.0,Panamerican Machinery,0.0,4.335526,76
2767,22602.0,Randy Rides Alone,0.0,4.315217,92


## Findings summary

**`movies_main.csv`** (45,486 rows)
- **Corrupt rows (3):** ids `1997-08-20`, `2012-09-29`, `2014-01-01` are columns shifted left by one — the whole row's fields are off by a position, and what should be `title` holds a comma-separated production-company list instead. These 3 rows need to be dropped or manually re-aligned in Task 5, not just null-filled.
- **Duplicates:** 6 exact-duplicate rows, 50 duplicate `id`s, 13 duplicate `title`+`release_date` pairs (the real duplicates — 3,188 duplicate-by-title-alone is mostly legitimate remakes/shared titles and is *not* a dedup signal on its own).
- **Date formats:** three formats in near-equal thirds — `YYYY-MM-DD` (15,194), `DD-MM-YYYY` (15,135), `MM/DD/YYYY` (15,067) — plus 90 nulls. Because `DD-MM-YYYY` and `MM/DD/YYYY` are ambiguous for day/month ≤ 12, a single `pd.to_datetime` call with `infer_datetime_format` will silently mis-parse a chunk of rows — needs explicit per-row format detection (see the `classify_date` regex above) rather than one global parse.
- **Budget:** 9.8% (4,437) are true nulls, but a much bigger issue is that **74.5% (33,887) are literal `0`**, clearly used as a missing-data placeholder rather than a real $0 budget. Combined, **84.3% of budget values are unusable as-is** — this is the single biggest data-quality problem in the dataset and should drive the cleaning strategy in Task 5 (e.g., treat 0 as null, consider imputation or exclusion for budget-dependent analysis).

**`ratings.json`** (9,067 records)
- Flat list of one dict per movie (not per individual rating event): `{movie_id, ratings_summary: {avg_rating, total_ratings, std_dev}, last_rated}`. `last_rated` is a Unix timestamp.
- All 9,067 `movie_id`s are unique and all are a subset of `movies_main.id` — no orphaned ratings.
- But ratings only cover **20% of the catalog** — 80% of movies (36,366) have no ratings record at all. Any ratings-based analysis in later tasks will only ever cover a fifth of the catalog; this should be called out explicitly wherever ratings drive a metric.

**`movie_extended.csv`** (45,466 rows)
- `genres` and `production_companies`: plain comma-separated strings (no semicolons found) — straightforward `str.split(",")` into a junction table.
- `production_countries` and `spoken_languages`: stringified **Python literals** (single-quoted dict keys), not valid JSON — `json.loads` will fail on these; use `ast.literal_eval`. Each is a list of `{iso_code, name}` dicts, so both need their own junction table (movie ↔ country, movie ↔ language).
- Null counts: `production_companies` 12,365 (27%), `genres` 2,442 (5%), `spoken_languages` 6. `production_countries` has no true nulls but 6,290 rows (14%) have an empty `[]` list — functionally the same as missing and should be treated as such.
- Same 3 corrupt ids as `movies_main.csv` show up here too, confirming the shift happened at the source, not during a later join.

**Cross-file**
- `id` is a clean, 1:1 shared join key between `movies_main.csv` and `movie_extended.csv` — no orphans either direction.
- `ratings.json`'s `movie_id` is always a subset of `movies_main.id`, but only covers ~20% of it (see above).

**Shortlist for Task 6 external validation** — movies with a high rating (≥4.2 avg, ≥50 ratings) but budget missing/zero, worth spot-checking against an external source (e.g. TMDB/IMDB) to tell "hidden gem we underfunded" from "bad data":
- id 1554 — *Down by Law* (4.49★, 311 ratings, budget $0)
- id 123505 — *A Modern Affair* (4.42★, 50 ratings, budget $0)
- id 41275 — *Python* (4.39★, 62 ratings, budget $0)
- id 112912 — *Why We Fight: The Battle of Russia* (4.39★, 135 ratings, budget $0)
- id 175027 — *Double Harness* (4.37★, 201 ratings, budget $0)
- (30 candidates total meet the threshold; see the full table above.)